# 03 — localized pages (fixture site build)

Builds the fixture site (netsnek.com on its jaen feature branch) and
inspects the generated pages: one variant per locale, correct
`<html lang>`, canonical links, unlocalized system routes, and no
German copy left in a translated page.

The build is expensive; set `JAEN_SKIP_SITE_BUILD=1` to reuse an
existing `public/`.


In [ ]:
import jaen_testkit as k
k.start_run('03-pages-i18n')
print(k.CONFIG['repo_root'])

In [ ]:
import os

SITE = k.CONFIG['site_dir']
PUBLIC = k.site_path('public')

with k.section('site build'):
    with k.check('fixture site builds') as c:
        if not os.path.isdir(SITE):
            c.skip('no fixture site checkout')
        if os.environ.get('JAEN_SKIP_SITE_BUILD') == '1':
            if os.path.isdir(PUBLIC):
                c.skip('JAEN_SKIP_SITE_BUILD=1 — reusing existing public/')
            c.skip('JAEN_SKIP_SITE_BUILD=1 but no public/ present')
        r = c.require(k.sh('yarn build', cwd=SITE,
                           timeout=k.CONFIG['build_timeout'],
                           label='gatsby build (site)'))
        c.ok('built in %.0fs' % r.duration_s)


In [ ]:
locales = k.CONFIG['site_locales']
default = k.CONFIG['site_default_locale']

with k.section('localized variants'):
    with k.check('index page exists per locale') as c:
        if not os.path.isdir(PUBLIC):
            c.skip('no public/ — build did not run')
        for locale in locales:
            path = (os.path.join(PUBLIC, 'index.html') if locale == default
                    else os.path.join(PUBLIC, locale, 'index.html'))
            c.expect_true(os.path.isfile(path), '%s -> %s' % (locale, os.path.relpath(path, PUBLIC)))

    with k.check('html lang matches the locale') as c:
        if not os.path.isdir(PUBLIC):
            c.skip('no public/')
        import re
        for locale in locales:
            path = (os.path.join(PUBLIC, 'index.html') if locale == default
                    else os.path.join(PUBLIC, locale, 'index.html'))
            html = k.read_text(path, '')
            m = re.search(r'<html[^>]*\blang=\"([^\"]+)\"', html)
            got = m.group(1) if m else None
            c.expect_true(bool(got) and got.lower().startswith(locale.split('-')[0]),
                          '%s: lang=%s' % (locale, got))


In [ ]:
with k.section('canonical + system routes'):
    with k.check('canonical link is absolute and normalized') as c:
        if not os.path.isdir(PUBLIC):
            c.skip('no public/')
        import re
        html = k.read_text(os.path.join(PUBLIC, 'index.html'), '')
        m = re.search(r'<link[^>]*rel=\"canonical\"[^>]*href=\"([^\"]+)\"', html)
        if not m:
            c.fail('no canonical link on the index page', abort=True)
        href = m.group(1)
        c.expect_true(href.startswith('https://'), href)
        c.expect_true('//' not in href.split('://', 1)[1], 'no double slashes: %s' % href)

    with k.check('system routes are not localized') as c:
        if not os.path.isdir(PUBLIC):
            c.skip('no public/')
        offenders = []
        for locale in locales:
            if locale == default:
                continue
            for system in ('cms', 'login', 'logout', 'settings', 'signup'):
                candidate = os.path.join(PUBLIC, locale, system)
                if os.path.isdir(candidate):
                    offenders.append('%s/%s' % (locale, system))
        c.expect_equal(offenders, [], 'no /<locale>/<system> directories')


In [ ]:
import html
import re

# German copy that must not survive into a translated page: the site's own
# sentences first, then the German function words as a generic net. A locale
# that still renders one of them in a text node, a visible attribute or its
# metadata was missed by the localization.
GERMAN_MARKERS = [
    'Wir verwirklichen', 'Experten aus unserem', 'Erzählen Sie uns',
    'Wir unterstützen', 'Wir lösen', 'Oder besuchen', 'Telefon',
    'Lassen Sie sich', 'Sie sind in guter', 'Werden Sie Teil',
    'Wir beraten', 'Wir entwickeln', 'Ihre Anfrage', 'Copyright ©'
]
GERMAN_WORDS = [
    'Wir', 'wir', 'Ihre', 'Ihr', 'Ihnen', 'Sie', 'uns', 'unser', 'unsere',
    'unserem', 'unseren', 'Unser', 'Unsere', 'und', 'oder', 'nicht', 'für',
    'mit', 'Idee', 'Know-How'
]

# Language-neutral by nature: the postal address, the legal copyright lines
# and the partner company names read the same in every locale.
LANGUAGE_NEUTRAL = [
    'Löwengasse 28 / Lokal 2A',
    'Copyright © 2024 Netsnek, Florian Herbert Kleber IT & Werbeagentur '
    'Nico Schett. All rights reserved.',
    'Copyright © 2023 Florian H. Kleber, Florian Herbert Kleber IT. '
    'All rights reserved.',
    'Werbeagentur Christian Aichner',
    'Florian Herbert Kleber IT',
    'Werbeagentur Nico Schett'
]

# German words that are also the correct word in a target language.
GERMAN_EXCEPTIONS = {'sl': {'Telefon'}}

_SCRIPT = re.compile(r'<(script|style|noscript)\b.*?</\1>', re.S | re.I)
_TAG = re.compile(r'<[^>]*>', re.S)
_TITLE = re.compile(r'<title[^>]*>(.*?)</title>', re.S | re.I)
_META = re.compile(r'<meta\b[^>]*>', re.I)
_ATTR = re.compile(r'\b(alt|placeholder|aria-label)\s*=\s*"([^"]*)"', re.I)
_WORD = r'(?<![^\W\d_])%s(?![^\W\d_])'


def visible_text(markup):
    """Everything a reader sees: metadata, visible attributes, text nodes."""
    body = _SCRIPT.sub(' ', markup)
    chunks = []
    for m in _TITLE.finditer(body):
        chunks.append(('<title>', html.unescape(m.group(1))))
    for m in _META.finditer(body):
        name = re.search(r'(?:name|property)\s*=\s*"([^"]*)"', m.group(0), re.I)
        content = re.search(r'content\s*=\s*"([^"]*)"', m.group(0), re.I)
        if name and content and re.search(r'description|title', name.group(1), re.I):
            chunks.append(('meta[%s]' % name.group(1),
                           html.unescape(content.group(1))))
    for m in _ATTR.finditer(body):
        chunks.append(('@%s' % m.group(1).lower(), html.unescape(m.group(2))))
    for raw in _TAG.split(body):
        chunks.append(('text', html.unescape(raw)))
    return [(where, re.sub(r'\s+', ' ', text).strip())
            for where, text in chunks if text.strip()]


def german_leftovers(path, locale):
    """German markers and function words left over in a translated page."""
    allowed = GERMAN_EXCEPTIONS.get(locale, set())
    hits = []
    for where, text in visible_text(k.read_text(path, '')):
        probe = text
        for phrase in LANGUAGE_NEUTRAL:
            probe = probe.replace(phrase, ' ')
        needles = [n for n in GERMAN_MARKERS + GERMAN_WORDS if n not in allowed]
        for needle in needles:
            # A single word is matched as a word, so an Italian "Telefono"
            # does not read as the German "Telefon".
            found = (needle in probe if ' ' in needle
                     else re.search(_WORD % re.escape(needle), probe))
            if found:
                hits.append('%s %r in %s' % (where, needle, text[:100]))
    return hits


with k.section('german leftovers'):
    for locale in locales:
        if locale == default:
            continue
        with k.check('%s pages carry no German copy' % locale) as c:
            if not os.path.isdir(PUBLIC):
                c.skip('no public/')
            root = os.path.join(PUBLIC, locale)
            if not os.path.isdir(root):
                c.fail('no %s/ in public/' % locale, abort=True)
            pages = []
            for folder, _dirs, files in os.walk(root):
                if 'index.html' in files:
                    pages.append(os.path.join(folder, 'index.html'))
            leftovers = []
            for page in sorted(pages):
                for hit in german_leftovers(page, locale):
                    leftovers.append('%s: %s'
                                     % (os.path.relpath(page, PUBLIC), hit))
            for hit in leftovers[:12]:
                c.note(hit)
            if len(leftovers) > 12:
                c.note('and %d more' % (len(leftovers) - 12))
            c.expect_equal(len(leftovers), 0,
                           '%d pages scanned, none German' % len(pages))

In [ ]:
k.summary()
k.save_results('results-03-pages-i18n.json')
rc = k.verdict()
assert rc == 0, 'run has FAILures — see the summary above'